# KMeans Clustering

Partition-based clustering that assigns n observations to k clusters by minimizing within-cluster variance (inertia). Simple, fast, and scalable — the most widely used clustering algorithm.

**When to Use:**
- Known or estimated number of clusters
- Roughly spherical, similarly-sized clusters expected
- Customer segmentation, grouping, anomaly pre-processing
- Medium-to-large datasets (scales well)
- As a baseline before trying density-based methods

**Key Assumptions / Considerations:**
- Clusters are convex and isotropic (spherical shape assumed)
- Number of clusters k must be specified in advance
- Sensitive to initialization — use KMeans++ (default) or run multiple times
- Sensitive to outliers — consider MiniBatchKMeans for large noisy data
- Features MUST be scaled (distance-based algorithm)
- High-dimensional data may need dimensionality reduction first

**References:**
- [sklearn KMeans](https://scikit-learn.org/stable/modules/generated/sklearn.cluster.KMeans.html)
- [sklearn Silhouette Score](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.silhouette_score.html)
- [sklearn Clustering Guide](https://scikit-learn.org/stable/modules/clustering.html)

In [ ]:
import numpy as np
import pandas as pd
import math
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, silhouette_samples, calinski_harabasz_score, davies_bouldin_score

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# test data

np.random.seed(100)
n_samples = 8000

# numeric features with cluster structure
age = np.concatenate([
    np.random.normal(25, 5, n_samples // 3),
    np.random.normal(45, 8, n_samples // 3),
    np.random.normal(65, 5, n_samples - 2 * (n_samples // 3))
]).astype(int)

income = np.concatenate([
    np.random.normal(30000, 5000, n_samples // 3),
    np.random.normal(60000, 10000, n_samples // 3),
    np.random.normal(90000, 8000, n_samples - 2 * (n_samples // 3))
])

num_transactions = np.concatenate([
    np.random.poisson(5, n_samples // 3),
    np.random.poisson(15, n_samples // 3),
    np.random.poisson(25, n_samples - 2 * (n_samples // 3))
])

# categorical
gender = np.random.choice(['Male', 'Female'], n_samples)
region = np.random.choice(['North', 'South', 'East', 'West'], n_samples)
product_type = np.random.choice(['A', 'B', 'C'], n_samples)

# ground truth labels (for validation only)
true_labels = np.concatenate([
    np.zeros(n_samples // 3),
    np.ones(n_samples // 3),
    np.full(n_samples - 2 * (n_samples // 3), 2)
]).astype(int)

# shuffle
idx = np.random.permutation(n_samples)
age = age[idx]
income = income[idx]
num_transactions = num_transactions[idx]
gender = gender[idx]
region = region[idx]
product_type = product_type[idx]
true_labels = true_labels[idx]

df = pd.DataFrame({
    'age': age,
    'income': income,
    'num_transactions': num_transactions,
    'gender': gender,
    'region': region,
    'product_type': product_type,
})

In [ ]:
# Load Data

#df = pd.read_csv("data.csv")

X = df.copy()

In [ ]:
# Data Overview

print(df.shape)
display(df.describe())

numeric_features = ['age', 'income', 'num_transactions']
categorical_features = ['gender', 'region', 'product_type']

fig, axes = plt.subplots(1, len(numeric_features), figsize=(5 * len(numeric_features), 4))
for i, col in enumerate(numeric_features):
    sns.histplot(df[col], kde=True, bins=30, ax=axes[i])
    axes[i].set_title(f"Distribution of {col}")
plt.tight_layout()
plt.show()

In [ ]:
# Preprocessing

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

X_processed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()
print(f"\u2705 Processed features: {len(feature_names)}")

In [ ]:
# Elbow Method

k_range = range(2, 11)
inertias = []
silhouette_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_processed)
    inertias.append(km.inertia_)
    silhouette_scores.append(silhouette_score(X_processed, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(k_range, inertias, 'bo-')
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("Inertia")
axes[0].set_title("Elbow Method")

axes[1].plot(k_range, silhouette_scores, 'ro-')
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score vs k")

plt.tight_layout()
plt.show()

best_k = k_range[np.argmax(silhouette_scores)]
print(f"\U0001f3c6 Best k by silhouette score: {best_k}")

In [ ]:
# Silhouette Diagram for best k

km_final = KMeans(n_clusters=best_k, random_state=42, n_init=10)
cluster_labels = km_final.fit_predict(X_processed)

sample_silhouette_values = silhouette_samples(X_processed, cluster_labels)
avg_score = silhouette_score(X_processed, cluster_labels)

fig, ax = plt.subplots(figsize=(10, 6))
y_lower = 10

for i in range(best_k):
    cluster_values = sample_silhouette_values[cluster_labels == i]
    cluster_values.sort()
    
    size = cluster_values.shape[0]
    y_upper = y_lower + size
    
    ax.fill_betweenx(np.arange(y_lower, y_upper), 0, cluster_values, alpha=0.7)
    ax.text(-0.05, y_lower + 0.5 * size, str(i))
    y_lower = y_upper + 10

ax.axvline(x=avg_score, color="red", linestyle="--", label=f"Avg: {avg_score:.3f}")
ax.set_xlabel("Silhouette Coefficient")
ax.set_ylabel("Cluster")
ax.set_title("Silhouette Diagram")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Cluster Evaluation

print("\u2705 Cluster Evaluation Metrics:")
print(f"   Silhouette Score:       {silhouette_score(X_processed, cluster_labels):.4f}")
print(f"   Calinski-Harabasz Score: {calinski_harabasz_score(X_processed, cluster_labels):.4f}")
print(f"   Davies-Bouldin Score:   {davies_bouldin_score(X_processed, cluster_labels):.4f}")

print(f"\n   Cluster sizes:")
unique, counts = np.unique(cluster_labels, return_counts=True)
for c, n in zip(unique, counts):
    print(f"   Cluster {c}: {n} samples ({n/len(cluster_labels)*100:.1f}%)")

In [ ]:
# Cluster Visualization (PCA)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_processed)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Predicted clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=cluster_labels, cmap='viridis', alpha=0.5, s=10)
centers_pca = pca.transform(km_final.cluster_centers_)
axes[0].scatter(centers_pca[:, 0], centers_pca[:, 1], c='red', marker='X', s=200, edgecolors='black')
axes[0].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[0].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[0].set_title("KMeans Clusters")
plt.colorbar(scatter1, ax=axes[0])

# Ground truth (if available)
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=true_labels, cmap='viridis', alpha=0.5, s=10)
axes[1].set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]:.1%} var)")
axes[1].set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]:.1%} var)")
axes[1].set_title("Ground Truth Labels")
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout()
plt.show()

In [ ]:
# Cluster Profiling

df_clustered = df.copy()
df_clustered['cluster'] = cluster_labels

# Numeric feature means per cluster
print("\u2705 Cluster Profiles (Numeric Features):")
profile = df_clustered.groupby('cluster')[numeric_features].mean()
display(profile)

# Heatmap of standardized cluster centers
scaler = StandardScaler()
profile_scaled = pd.DataFrame(
    scaler.fit_transform(profile),
    columns=numeric_features,
    index=profile.index
)

plt.figure(figsize=(10, 5))
sns.heatmap(profile_scaled.T, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Cluster Centers (Standardized)")
plt.xlabel("Cluster")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

# Categorical distribution per cluster
for col in categorical_features:
    print(f"\n--- {col} ---")
    ct = pd.crosstab(df_clustered['cluster'], df_clustered[col], normalize='index')
    display(ct)

In [ ]:
# Cluster Size Distribution

plt.figure(figsize=(8, 4))
sns.countplot(x=cluster_labels)
plt.xlabel("Cluster")
plt.ylabel("Count")
plt.title("Cluster Size Distribution")
plt.tight_layout()
plt.show()

In [ ]:
# future work:
#   MiniBatchKMeans for very large datasets (faster, similar quality)
#   KMeans++ vs random initialization comparison
#   t-SNE or UMAP for 2D visualization (better non-linear projection than PCA)
#   feature engineering to improve cluster separation
#   hierarchical clustering (AgglomerativeClustering) comparison
#   Gaussian Mixture Models for soft cluster assignments